# Melbourne Housing Price Prediction - Decision Tree Model
This notebook implements a Decision Tree Regressor for Melbourne house prices

In [14]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

Libraries imported successfully!


In [15]:
# Load Melbourne Housing Data
melb_data = pd.read_csv("melb_data.csv")
print(f"Dataset shape: {melb_data.shape}")
melb_data.head()

Dataset shape: (13580, 21)


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [16]:
# Check for missing values in our selected features
features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']

print("Missing values per column:")
print(melb_data[features].isnull().sum())

Missing values per column:
Rooms         0
Bathroom      0
Landsize      0
Lattitude     0
Longtitude    0
dtype: int64


In [17]:
# Handle missing values by dropping rows with missing values in our features
melbourne_model_data = melb_data.dropna(subset=features)
print(f"Filtered data shape: {melbourne_model_data.shape}")

Filtered data shape: (13580, 21)


In [18]:
# Define target and features
y = melbourne_model_data.Price
X = melbourne_model_data[features]

print(f"Target variable (Price) shape: {y.shape}")
print(f"Features shape: {X.shape}")

Target variable (Price) shape: (13580,)
Features shape: (13580, 5)


In [19]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42)


print (f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

Training set size: 10185
Validation set size: 3395


In [20]:
# Hyperparameter Tuning - Find optimal max_leaf_nodes
def get_mae(max_leaf_nodes, X_train, X_val, y_train, y_val):
    """Calculate MAE for different max_leaf_nodes values"""
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return mean_absolute_error(y_val, y_pred)

# Test different max_leaf_nodes values
max_leaf_nodes_options = [5, 10, 25, 50, 100, 150, 200, 250]
mae_results = []

for nodes in max_leaf_nodes_options:
    mae = get_mae(nodes, X_train, X_val, y_train, y_val)
    mae_results.append({'max_leaf_nodes': nodes, 'MAE': mae})
    print(f"max_leaf_nodes={nodes}: MAE = ${mae:,.2f}")

max_leaf_nodes=5: MAE = $353,119.21
max_leaf_nodes=10: MAE = $311,108.85
max_leaf_nodes=25: MAE = $281,129.60
max_leaf_nodes=50: MAE = $255,724.28
max_leaf_nodes=100: MAE = $240,630.81
max_leaf_nodes=150: MAE = $232,630.97
max_leaf_nodes=200: MAE = $232,927.91
max_leaf_nodes=250: MAE = $229,236.86


In [21]:
# Find best max_leaf_nodes
results_df = pd.DataFrame(mae_results)
best_idx = results_df['MAE'].idxmin()
best_max_leaf_nodes = results_df.loc[best_idx, 'max_leaf_nodes']
best_mae = results_df.loc[best_idx, 'MAE']

print(f"\nBest max_leaf_nodes: {best_max_leaf_nodes}")
print(f"Best MAE: ${best_mae:,.2f}")


Best max_leaf_nodes: 250
Best MAE: $229,236.86


In [22]:
# Train final model with optimal hyperparameters
final_model = DecisionTreeRegressor(
    max_leaf_nodes=best_max_leaf_nodes,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final Decision Tree model trained!")
print(final_model)

Final Decision Tree model trained!
DecisionTreeRegressor(max_leaf_nodes=np.int64(250), random_state=42)


In [23]:
# Final model evaluation
final_train_pred = final_model.predict(X_train)
final_val_pred = final_model.predict(X_val)

final_train_mae = mean_absolute_error(y_train, final_train_pred)
final_val_mae = mean_absolute_error(y_val, final_val_pred)

print("Final Model Performance:")
print(f"Training MAE: ${final_train_mae:,.2f}")
print(f"Validation MAE: ${final_val_mae:,.2f}")
print(f"\nThe model predicts Melbourne house prices with an average error of ~${final_val_mae/1000:,.0f}K")

Final Model Performance:
Training MAE: $178,237.94
Validation MAE: $229,236.86

The model predicts Melbourne house prices with an average error of ~$229K


In [24]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': final_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Decision Tree Feature Importances:")
for idx, row in feature_importance.iterrows():
    print(f"{row['Feature']}: {row['Importance']:.3f}")

Decision Tree Feature Importances:
Longtitude: 0.296
Rooms: 0.270
Lattitude: 0.244
Landsize: 0.148
Bathroom: 0.043


In [25]:
# Summary
print("=" * 50)
print("SUMMARY - Decision Tree Model for Melbourne Housing")
print("=" * 50)
print(f"Model Type: {type(final_model).__name__}")
print(f"Optimal max_leaf_nodes: {best_max_leaf_nodes}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Training MAE: ${final_train_mae:,.2f}")
print(f"Validation MAE: ${final_val_mae:,.2f}")
print(f"Most important feature: {feature_importance.iloc[0]['Feature']} ({feature_importance.iloc[0]['Importance']:.3f})")
print(f"Second most important: {feature_importance.iloc[1]['Feature']} ({feature_importance.iloc[1]['Importance']:.3f})")

SUMMARY - Decision Tree Model for Melbourne Housing
Model Type: DecisionTreeRegressor
Optimal max_leaf_nodes: 250
Training samples: 10185
Validation samples: 3395
Training MAE: $178,237.94
Validation MAE: $229,236.86
Most important feature: Longtitude (0.296)
Second most important: Rooms (0.270)
